In this script, we demonstrate an example of what happens when we do not include a necessary transformation in TFP.

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
tfd = tfp.distributions
tfb = tfp.bijectors

In [ ]:
alpha_prior, beta_prior = 4, 3
prior_dist = tfd.Beta(alpha_prior, beta_prior)

conditioned_log_prob = lambda z: prior_dist.log_prob(z)
tf.random.set_seed(20)

# TFP's default Adam (learning rate 1e-3) drifts slowly, so the checkpoints span 5,000 to 300,000 iterations
CHECKPOINTS = [5_000, 20_000, 100_000, 300_000]


def run_fit(surr_post, num_steps):
    """Fit with TFP's default Adam and return the base Normal's (loc, scale) after every step."""
    def base_normal():
        # look the base Normal up on every step: a transformed surrogate rebuilds it from its
        # variables on each access, so a reference taken once would keep the initial values
        base = surr_post
        while hasattr(base, "distribution"):
            base = base.distribution
        return base
    return tfp.vi.fit_surrogate_posterior(
        conditioned_log_prob,
        surrogate_posterior=surr_post,
        optimizer=tf.optimizers.Adam(),
        num_steps=num_steps,
        sample_size=100,
        trace_fn=lambda traceable_quantities: (base_normal().loc, base_normal().scale),
    )


def at_checkpoints(trace):
    """(loc, scale) after each number of iterations in CHECKPOINTS."""
    locs, scales = (np.asarray(t) for t in trace)
    return {n: (float(locs[n - 1]), float(scales[n - 1])) for n in CHECKPOINTS}


# one fit of an untransformed Normal surrogate, read off at each checkpoint
untransformed = at_checkpoints(run_fit(tfd.Normal(loc=tf.Variable(0.5), scale=tf.Variable(0.25)), max(CHECKPOINTS)))
for n, (loc, scale) in untransformed.items():
    print(f"after {n:>7,} iterations: Normal({loc:.3g}, {scale:.3g}^2)")

x = np.linspace(-1, 2, 500)
pdf_prior = prior_dist.prob(x)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import ScalarFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, ax = plt.subplots(figsize=(12, 6))

FONT_SIZE = 25
mpl.rcParams['font.size'] = 20
STYLES = ["-", "--", "-.", ":"]
COLORS = ["blue", "orange", "green", "red"]

# ---- main plot
for (n, (loc, scale)), ls, color in zip(untransformed.items(), STYLES, COLORS):
    ax.plot(x, tfd.Normal(loc, scale).prob(x), ls=ls, color=color,
            label=f"after {n:,} iterations:\n Normal({loc:.3g}, {scale:.3g}$^2$)")
ax.plot(x, pdf_prior, '--', color='black', label=f'PDF of prior beta({alpha_prior}, {beta_prior})')
ax.axvspan(x.min(), 0, color='blue', alpha=0.15, label='Out of Support for Beta')
ax.axvspan(1, x.max(), color='blue', alpha=0.15)

ax.set_title('TFP Unconstrained Normal Fit to Beta Prior', fontsize=FONT_SIZE)
ax.set_xlabel('x', fontsize=FONT_SIZE)
ax.set_ylabel('Density', fontsize=FONT_SIZE)
ax.grid(True)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=FONT_SIZE-5)

# ---- inset (upper left): zoomed out to show the last two fits, which are too wide for the main axes
axins = inset_axes(ax, width="35%", height="60%", loc="upper left", borderpad=3.0)
last_loc, last_scale = untransformed[CHECKPOINTS[-1]]
inset_x = np.linspace(last_loc - 4 * last_scale, last_loc + 4 * last_scale, 2000)
for (n, (loc, scale)), ls, color in list(zip(untransformed.items(), STYLES, COLORS))[-2:]:
    axins.plot(inset_x, tfd.Normal(loc, scale).prob(inset_x), color=color, ls=ls)
axins.grid(True)
formatter = ScalarFormatter(useMathText=True)  # scientific notation for the inset's y-axis
formatter.set_scientific(True)
formatter.set_powerlimits((0, 0))
axins.yaxis.set_major_formatter(formatter)
plt.show()


What happens when we tell TFP to use a transformation?

In [ ]:
def make_sigmoid_surr_approx(loc, scale):
    return tfp.experimental.vi.build_factored_surrogate_posterior(
        event_shape=(),
        bijector=tfb.Sigmoid(),
        initial_parameters={'loc': loc, 'scale': scale}
    )

# one fit of the sigmoid-transformed surrogate, read off at the same checkpoints
transformed = at_checkpoints(run_fit(make_sigmoid_surr_approx(0.5, 0.25), max(CHECKPOINTS)))
for n, (loc, scale) in transformed.items():
    print(f"after {n:>7,} iterations: latent Normal({loc:.3g}, {scale:.3g}^2)")


In [ ]:
# the sigmoid-transformed fits at each checkpoint

plt.figure(figsize=(12, 8))
x = np.linspace(0, 1, 500)
for n, (loc, scale) in transformed.items():
    plt.plot(x, make_sigmoid_surr_approx(loc, scale).prob(x).numpy(), label=f'Sigmoid Normal PDF {n:,} iter', linewidth=2)

# Plot the true Beta distribution we're trying to approximate
true_distribution = prior_dist
plt.plot(x, true_distribution.prob(x).numpy(), '--', label=f'True Beta({alpha_prior}, {beta_prior}) Distribution', linewidth=3, color='black')

plt.title('TFP Sigmoid Transformed Normal Approximations to Beta Distribution')
plt.xlabel('x')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


Can NumPyro defaults produce the correct transformation output?

(the following is copy-pasted from beta_numpyro.ipynb notebook under 1dbeta_binomial)

In [ ]:
import json
import os
import numpy as np
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, TraceMeanField_ELBO
from numpyro.infer.autoguide import AutoNormal
from numpyro.optim import Adam
import matplotlib.pyplot as plt
from tqdm import tqdm

from scipy.special import expit  # sigmoid

In [ ]:
# helpers
def binomial(n_samples, theta_like=0.7, seed=20):
    np.random.seed(seed)
    data = np.random.binomial(n_samples, theta_like)
    return data

def _logistic_normal_mean_std_mc(mu, sigma, n_samples=10_000, seed=0):
    mu = float(np.squeeze(mu))
    sigma = float(np.squeeze(sigma))
    rng = np.random.default_rng(seed)
    z = rng.normal(loc=mu, scale=sigma, size=n_samples)
    theta = expit(z)
    return float(theta.mean()), float(theta.std(ddof=1))

def _trace_locscale_to_theta_moments(loc_trace, scale_trace, n_samples=10_000, seed=0):
    loc_trace = np.asarray(loc_trace, dtype=float)
    scale_trace = np.asarray(scale_trace, dtype=float)
    out_mean = np.empty(loc_trace.shape[0], dtype=float)
    out_std  = np.empty(loc_trace.shape[0], dtype=float)
    for i in range(loc_trace.shape[0]):
        out_mean[i], out_std[i] = _logistic_normal_mean_std_mc(
            loc_trace[i], scale_trace[i], n_samples=n_samples, seed=seed
        )
    return out_mean, out_std



alpha_prior, beta_prior = 4, 3
theta_like = 0.7 # this does not matter since we have no data
data = 0
n_samples = 0
max_iters = 100_000
adam_step = 5e-4

In [ ]:
# run in numpyro 
def model(y):
    theta = numpyro.sample("theta", dist.Beta(alpha_prior, beta_prior))
    numpyro.sample("obs", dist.Binomial(n_samples, theta), obs=y)
y_data = jnp.array(data)
optimizer = Adam(step_size=adam_step)
elbo = TraceMeanField_ELBO(num_particles=100)
guide = AutoNormal(model)
svi = SVI(model, guide, optimizer, loss=elbo)

# we're going to run without tracking, so we can use
# the built-in SVI loop, which is faster than our custom one

svi_result = svi.run(jax.random.PRNGKey(0), max_iters, data)
params = svi_result.params
output_dist = make_sigmoid_surr_approx(params['theta_auto_loc'], params['theta_auto_scale'])


In [ ]:
plt.figure(figsize=(12, 8))
x = np.linspace(0, 1, 500)  
plt.plot(x, output_dist.prob(x).numpy(), label=f'Sigmoid Normal fit by NumPyro after 100k iter', linewidth=2)

# Plot the true Beta distribution we're trying to approximate
true_distribution = prior_dist  
plt.plot(x, true_distribution.prob(x).numpy(), '--', label=f'True Beta({alpha_prior}, {beta_prior}) Distribution', linewidth=3, color='black')

plt.title('NumPyro Sigmoid Transformed Normal Approximation to Beta Distribution')
plt.xlabel('x')
plt.ylabel('Probability Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()